# SPR-03 — Model Training + Ensembling

**Ticket:** SPR-03
**Owner:** Sakhiur
**Depends on:** SPR-02 (resampled train set)
**Folder:** `notebooks/04_modeling/02_ensembling.ipynb`
**Output:** trained models saved to `models/{decision_tree,logistic_regression,random_forest,xgboost}/`, metrics logged to `results/tables/`

**Inspiration:** `SOTA_Paper_8.ipynb` cells 37-48 (hyperparameter search + VotingClassifier hard/soft voting), `cross-validation.ipynb` cells 37/40 (Stratified K-Fold evaluation loop). Adapted for multi-class `income_class` and depth-capped models to avoid the Colab stall documented earlier in the project.

## 1. Load resampled train set from SPR-02

In [3]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_train_resampled_Sakhiur.csv

--2026-08-26 05:50:13--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_train_resampled_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.6, 3.171.171.128, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a7c7183f2c81824bdfb6fa9/9f6172a1773ccf26810ab1921082ea2b55dd1e0849d93107fe3e14ac836e8fd0?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27X_train_resampled_Sakhiur.csv%3B+filename%3D%22X_train_resampled_Sakhiur.csv%22%3B&X-Xet-Cas-Uid=public&user_id=public&response-content-type=text%2Fcsv&Expires=1787727013&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE3YzcxODNmMmM4MTgyNGJkZmI2ZmE5LzlmNjE3MmExNzczY2NmMjY4MTBhYjE5MjEwODJlYTJiNTVkZDFlMDg0OWQ5MzEwN2ZlM2UxNGFjODM2ZThmZDBcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlc

In [4]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_test_Sakhiur.csv

--2026-08-26 05:50:14--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_test_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.6, 3.171.171.128, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/X_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2FX_test_Sakhiur.csv=&etag=%2249ac492745f5c946abd59d29947ad5baa2a6d39d%22 [following]
--2026-08-26 05:50:14--  https://huggingface.co/api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/X_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2FX_test_Sakhiur.csv=&etag=%2249ac492745f5c946abd59d29947ad5baa2a6d39d%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 10184720 (9.7M) [text/plain]
Saving t

In [5]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_train_resampled_Sakhiur.csv

--2026-08-26 05:50:15--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_train_resampled_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.6, 3.171.171.128, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_train_resampled_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_train_resampled_Sakhiur.csv=&etag=%22d1dadfe2ca86451459223e55ad3fb4a057c0f083%22 [following]
--2026-08-26 05:50:15--  https://huggingface.co/api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_train_resampled_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_train_resampled_Sakhiur.csv=&etag=%22d1dadfe2ca86451459223e55ad3fb4a057c0f083%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response.

In [6]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_test_Sakhiur.csv

--2026-08-26 05:50:15--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_test_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.6, 3.171.171.128, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_test_Sakhiur.csv=&etag=%2265b12cabc0360608ebbdad6a81d0164f0522455e%22 [following]
--2026-08-26 05:50:15--  https://huggingface.co/api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_test_Sakhiur.csv=&etag=%2265b12cabc0360608ebbdad6a81d0164f0522455e%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1199517 (1.1M) [text/plain]
Saving to

In [7]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/final_dataset_preprocessed_distributed_social_class.csv

--2026-08-26 05:50:15--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/final_dataset_preprocessed_distributed_social_class.csv
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.6, 3.171.171.128, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a7c7183f2c81824bdfb6fa9/e667362f3c877b4bc4e46b95a9b7382d66b4e3037af217c9b916437c0c549fcc?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27final_dataset_preprocessed_distributed_social_class.csv%3B+filename%3D%22final_dataset_preprocessed_distributed_social_class.csv%22%3B&X-Xet-Cas-Uid=public&response-content-type=text%2Fcsv&user_id=public&Expires=1787727015&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE3YzcxODNmMmM4MTgyNGJkZmI2ZmE5L2U2NjczNjJmM2M4NzdiNGJjNGU0NmI5NWE5YjczODJkNjZiNGUzMDM3YWYyMTdjOWI5MTY0MzdjMGM1NDlmY2N

In [8]:
import pandas as pd
import numpy as np
import random
import joblib
import os

np.random.seed(42)
random.seed(42)

X_train = pd.read_csv('X_train_resampled_Sakhiur.csv')
y_train = pd.read_csv('y_train_resampled_Sakhiur.csv').iloc[:, 0]
X_test = pd.read_csv('X_test_Sakhiur.csv')
y_test = pd.read_csv('y_test_Sakhiur.csv').iloc[:, 0]

print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (1049480, 25) Test: (150325, 25)


In [9]:
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Load the final, cleaned, NOT-yet-resampled dataset
df = pd.read_csv('final_dataset_preprocessed_distributed_social_class.csv')

TARGET_COL = 'income_class'

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Same split params as your original SPR-02 split, so this reproduces the exact same
# train/test partition, just without SMOTE applied to X_train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('X_train (pre-SMOTE):', X_train.shape)
print('X_test:', X_test.shape)
print()
print('y_train class distribution (should match your real, unresampled proportions):')
print(y_train.value_counts())
print()
print('y_test class distribution:')
print(y_test.value_counts())

X_train (pre-SMOTE): (601297, 25)
X_test: (150325, 25)

y_train class distribution (should match your real, unresampled proportions):
income_class
No_income    262370
Lower        191540
Middle       140726
Upper          6661
Name: count, dtype: int64

y_test class distribution:
income_class
No_income    65593
Lower        47885
Middle       35182
Upper         1665
Name: count, dtype: int64


In [10]:
X_test_saved = pd.read_csv('X_test_Sakhiur.csv')
print('Matches saved X_test:', X_test.reset_index(drop=True).equals(X_test_saved.reset_index(drop=True)))

Matches saved X_test: True


In [11]:
!pip install catboost -q


## 1b. Encode target labels
XGBoost requires integer class labels (0,1,2,...), not strings. This encodes `y_train`/`y_test` once here, and every downstream cell decodes predictions back to real class names before printing or plotting.

In [12]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print('Class mapping:')
for i, label in enumerate(label_encoder.classes_):
    print(f'  {i} -> {label}')

Class mapping:
  0 -> Lower
  1 -> Middle
  2 -> No_income
  3 -> Upper


## 2. Hyperparameter search space
uses the five models(decision tree, RF, XGBoost, logistic regression, and CatBoost) and depth-capped to stay inside Colab free-tier limits, per the compute constraint already flagged for this project.

In [13]:
param_distributions = {
    'DecisionTreeClassifier': {
        'max_depth': [5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'RandomForestClassifier': {
        'n_estimators': [100, 150, 200],
        'max_depth': [10],  # capped per project compute constraint, do not widen
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'XGBClassifier': {
        'n_estimators': [100, 150, 200],
        'max_depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1]
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1.0, 10.0],
        'max_iter': [1000]
    },
    'CatBoostClassifier': {
        'iterations': [200, 400, 600],
        'depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [3, 5, 7]
    },
    'LGBMClassifier': {
        'n_estimators': [100, 200, 300],
        'max_depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [20, 31, 40]
    }
}

## 3. Randomized search per model

## 3b. Manual resume: paste known best params to skip search
If Colab disconnects mid-search, or you already found good params in a previous run, paste them into `KNOWN_BEST_PARAMS` below. Any model listed here skips `RandomizedSearchCV` entirely and is just fit directly. Any model NOT listed here still goes through the normal search in the next cell.

Leave a model's entry out (or set the whole dict empty) to search everything from scratch.

In [14]:
# Fill in only the models you already have good params for. Leave others out entirely,
# they'll be searched normally below. Example after a DT+RF search already succeeded:
#
# KNOWN_BEST_PARAMS = {
#     'DecisionTreeClassifier': {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 20},
#     'RandomForestClassifier': {'n_estimators': 100, 'min_samples_split': 10,
#                                 'min_samples_leaf': 4, 'max_depth': 10},
# }

KNOWN_BEST_PARAMS = {

}

if KNOWN_BEST_PARAMS:
    print('Will skip search for:', list(KNOWN_BEST_PARAMS.keys()))
else:
    print('No manual params provided, everything below will be searched fresh.')

No manual params provided, everything below will be searched fresh.


In [ ]:
!pip install catboost lightgbm imbalanced-learn -q

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


RESAMPLER = SMOTE(random_state=42)
N_SPLITS = 5  # number of cross-validation folds, stated once here so it's easy to report accurately

base_models = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'XGBClassifier': XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'LogisticRegression': LogisticRegression(random_state=42),
    'CatBoostClassifier': CatBoostClassifier(random_seed=42, loss_function='MultiClass', verbose=0),
    'LGBMClassifier': LGBMClassifier(random_state=42, objective='multiclass')
}

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
best_params = {}
best_estimators = {}
cv_score_summary = {}  # holds mean, std, and n_folds per model

model_classes = {
    'DecisionTreeClassifier': DecisionTreeClassifier,
    'RandomForestClassifier': RandomForestClassifier,
    'XGBClassifier': XGBClassifier,
    'LogisticRegression': LogisticRegression,
    'CatBoostClassifier': CatBoostClassifier,
    'LGBMClassifier': LGBMClassifier
}
model_fixed_kwargs = {
    'DecisionTreeClassifier': {'random_state': 42},
    'RandomForestClassifier': {'random_state': 42},
    'XGBClassifier': {'random_state': 42, 'eval_metric': 'mlogloss'},
    'LogisticRegression': {'random_state': 42},
    'CatBoostClassifier': {'random_seed': 42, 'loss_function': 'MultiClass', 'verbose': 0},
    'LGBMClassifier': {'random_state': 42, 'objective': 'multiclass'}
}

print(f'Cross-validation: {N_SPLITS}-fold StratifiedKFold, resampling applied fresh inside each fold\n')

for name, model in base_models.items():
    if name in KNOWN_BEST_PARAMS:
        print(f'Skipping search for {name}, using provided params...')
        params = KNOWN_BEST_PARAMS[name]
        X_res, y_res = RESAMPLER.fit_resample(X_train, y_train_enc)
        fitted = model_classes[name](**params, **model_fixed_kwargs[name])
        fitted.fit(X_res, y_res)
        best_params[name] = params
        best_estimators[name] = fitted
        print(f'  Used params: {params}')
        print(f'  (No CV mean/std available, params were reused directly, not searched)\n')
        continue

    print(f'Searching {name}...')
    imb_pipeline = ImbPipeline([
        ('resample', RESAMPLER),
        ('clf', model)
    ])

    prefixed_params = {f'clf__{k}': v for k, v in param_distributions[name].items()}

    search = RandomizedSearchCV(
        imb_pipeline, prefixed_params,
        n_iter=10, cv=cv, scoring='f1_macro',
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train_enc)

    best_estimators[name] = search.best_estimator_.named_steps['clf']
    best_params[name] = {k.replace('clf__', ''): v for k, v in search.best_params_.items()}

    best_idx = search.best_index_
    mean_score = search.cv_results_['mean_test_score'][best_idx]
    std_score = search.cv_results_['std_test_score'][best_idx]
    cv_score_summary[name] = {'mean': mean_score, 'std': std_score, 'n_folds': N_SPLITS}

    print(f'  Best params: {best_params[name]}')
    print(f'  Best CV macro F1 ({N_SPLITS}-fold): {mean_score*100:.2f}% ± {std_score*100:.2f}%\n')

print(f'=== CV macro F1 summary ({N_SPLITS}-fold cross-validation) ===')
for name, scores in cv_score_summary.items():
    print(f'{name}: {scores["mean"]*100:.2f}% ± {scores["std"]*100:.2f}%')

Cross-validation: 5-fold StratifiedKFold, resampling applied fresh inside each fold

Searching DecisionTreeClassifier...


## 4. Evaluate each tuned model on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score,accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

test_results = {}
for name, model in best_estimators.items():
    y_train_pred_enc = model.predict(X_train)
    y_train_pred = label_encoder.inverse_transform(y_train_pred_enc)
    train_acc = accuracy_score(y_train, y_train_pred)

    y_pred_enc = model.predict(X_test)
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    test_acc = accuracy_score(y_test, y_pred)

    gap = train_acc - test_acc
    print(f'{name}: Train acc = {train_acc:.4f}, Test acc = {test_acc:.4f}, Gap = {gap:.4f}')

    macro_f1 = f1_score(y_test, y_pred, average='macro')
    test_results[name] = macro_f1
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, digits=3))
    cm = confusion_matrix(y_test, y_pred, labels=label_encoder.classes_)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.show()

## 5. Ensembling: hard + soft voting
`voting='soft'` requires every base estimator to support `predict_proba`, which all six models here do.

In [ ]:
from sklearn.ensemble import VotingClassifier

# Rank models by CV macro F1 (already computed during the search loop above)
# Falls back gracefully for any model that used KNOWN_BEST_PARAMS and has no CV score
ranked_models = sorted(
    cv_score_summary.items(),
    key=lambda x: x[1]['mean'],
    reverse=True
)

if len(ranked_models) < 3:
    print(f'Warning: only {len(ranked_models)} models have CV scores '
          f'(others used KNOWN_BEST_PARAMS with no CV run). '
          f'Add cross_val_score for those manually if you want them eligible for top-3.')

top_3_names = [name for name, _ in ranked_models[:3]]
print(f'Top 3 models by CV macro F1: {top_3_names}')
for name, scores in ranked_models[:3]:
    print(f'  {name}: {scores["mean"]*100:.2f}% ± {scores["std"]*100:.2f}%')

top_3_estimators = [(name, best_estimators[name]) for name in top_3_names]

voting_clf_hard = VotingClassifier(estimators=top_3_estimators, voting='hard')
voting_clf_soft = VotingClassifier(estimators=top_3_estimators, voting='soft')

print('\nTraining hard voting ensemble (top 3)...')
voting_clf_hard.fit(X_train, y_train_enc)
print('Training soft voting ensemble (top 3)...')
voting_clf_soft.fit(X_train, y_train_enc)

## 6. Evaluate ensembles

In [ ]:
for clf, label in zip([voting_clf_hard, voting_clf_soft], ['Hard Voting', 'Soft Voting']):
    y_train_pred_enc = clf.predict(X_train)
    y_train_pred = label_encoder.inverse_transform(y_train_pred_enc)
    train_acc = accuracy_score(y_train, y_train_pred)
    y_pred_enc = clf.predict(X_test)
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    test_acc = accuracy_score(y_test, y_pred)
    gap = train_acc - test_acc
    print(f'{label}: Train acc = {train_acc:.4f}, Test acc = {test_acc:.4f}, Gap = {gap:.4f}')
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    test_results[label] = macro_f1
    print(f'--- {label} ---')
    print(classification_report(y_test, y_pred, digits=3))
    cm = confusion_matrix(y_test, y_pred, labels=label_encoder.classes_)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix: {label}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.show()

print('\nFinal comparison (macro F1):')
for name, score in sorted(test_results.items(), key=lambda x: -x[1]):
    print(f'{name}: {score:.4f}')

## 7. Save models and metrics
This is the handoff point for SPR-04 (figures) and SPR-05 (SHAP), both of which read these saved models read-only and must not retrain them.

In [ ]:
os.makedirs('models/decision_tree', exist_ok=True)
os.makedirs('models/random_forest', exist_ok=True)
os.makedirs('models/xgboost', exist_ok=True)
os.makedirs('models/logistic_regression', exist_ok=True)
os.makedirs('models/catboost', exist_ok=True)
os.makedirs('models/lightgbm', exist_ok=True)

joblib.dump(best_estimators['DecisionTreeClassifier'], 'models/decision_tree/model.joblib')
joblib.dump(best_estimators['RandomForestClassifier'], 'models/random_forest/model.joblib')
joblib.dump(best_estimators['XGBClassifier'], 'models/xgboost/model.joblib')
joblib.dump(best_estimators['LogisticRegression'], 'models/logistic_regression/model.joblib')
joblib.dump(best_estimators['CatBoostClassifier'], 'models/catboost/model.joblib')
joblib.dump(best_estimators['LGBMClassifier'], 'models/lightgbm/model.joblib')
joblib.dump(voting_clf_soft, 'models/voting_ensemble_soft.joblib')
joblib.dump(voting_clf_hard, 'models/voting_ensemble_hard.joblib')

results_df = pd.DataFrame(list(test_results.items()), columns=['model', 'macro_f1'])
results_df = results_df.sort_values('macro_f1', ascending=False)
results_df.to_csv('results/tables/model_comparison.csv', index=False)
print(results_df)
print('\nBest model for SPR-05 SHAP analysis:', results_df.iloc[0]['model'])

# Save the label encoder so SPR-04 / SPR-05 can decode predictions back to class names
joblib.dump(label_encoder, 'models/label_encoder.joblib')
print('Saved models/label_encoder.joblib')